# Aula 04 - Da Nuvem à Fronteira: Monte Carlo e Otimização de Markowitz

Módulo 08 AdmTech, Computação · Semana 6 · Sprint 3 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula04.ipynb)

## O que este notebook é

Laboratório executável da Aula 04, que alimenta o artefato A3 (Motor de Simulação de Monte Carlo) e prepara o A4 (Otimizador de Carteira com Fronteira Eficiente). Roda no Colab pelo link acima ou localmente a partir do repositório clonado. Sem rede, a carga usa o snapshot versionado em `dados/`; com `M08_OFFLINE=1`, a queda para a rede fica desligada, como no CI.

## 1. Retornos, retorno esperado, covariância e CDI

A amostra segue o tratamento de ausência único do módulo (ADR-009, a mesma regra de `tools/retornos.py`). A lacuna isolada de 24/10/2025 (BOVA11, IVVB11 e GOLD11) é preenchida com o último preço, como no exemplo de ADR da Aula 01, e o retorno desse pregão fica zero nos três ETFs. Nada é preenchido antes da primeira cotação, e a análise conjunta usa a janela comum dos 15 ativos: como a GOLD11 só é cotada a partir de 11/12/2020, são 1.423 retornos diários, de 14/12/2020 a 31/08/2026, a mesma janela da matriz da Aula 03.

O custo da janela comum é perder março de 2020: a volatilidade anual da PETR4 vai de 39,25% na série inteira, a da Aula 02, para 33,40% na janela comum. A anualização é por 252 pregões, como na Aula 02, e a taxa livre de risco é o CDI acumulado na mesma janela.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"
PREGOES_ANO = 252


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab."""
    local = os.path.join(DADOS_LOCAL, nome)
    if os.path.exists(local):
        return pd.read_csv(local, **kwargs)
    if os.environ.get("M08_OFFLINE") == "1":
        raise FileNotFoundError("M08_OFFLINE=1 e %s não existe no disco" % local)
    return pd.read_csv(DADOS_BRUTOS + nome, **kwargs)


# ADR-009: ffill preenche só a lacuna isolada (nunca antes da primeira cotação), e
# o dropna final deixa os pregões em que os 15 ativos têm retorno, a janela comum.
precos = ler_csv("precos_ajustados.csv", index_col="data", parse_dates=["data"]).ffill()
retornos_por_ativo = precos.pct_change(fill_method=None).iloc[1:]
retornos = retornos_por_ativo.dropna()
mu = retornos.mean() * PREGOES_ANO
cov = retornos.cov() * PREGOES_ANO
cdi = ler_csv("cdi_diario.csv", index_col="data", parse_dates=["data"])["valor"]
janela = cdi.loc[retornos.index[0]:retornos.index[-1]]
rf = float((1 + janela / 100).prod() ** (PREGOES_ANO / len(janela)) - 1)
TICKERS = list(retornos.columns)

print("retornos diários:", len(retornos), "de", retornos.index[0].date(), "a", retornos.index[-1].date())
print("PETR4, volatilidade anual: série inteira %.2f%%, janela comum %.2f%%"
      % (100 * retornos_por_ativo["PETR4"].std() * np.sqrt(PREGOES_ANO),
         100 * retornos["PETR4"].std() * np.sqrt(PREGOES_ANO)))
print("CDI anualizado na janela: %.2f%%" % (100 * rf))
pd.DataFrame({"retorno": mu, "volatilidade": np.sqrt(np.diag(cov))}, index=TICKERS).mul(100).round(2).sort_values("retorno")

## 2. Pesos, retorno e risco em forma vetorial

Uma carteira é um vetor `w` de pesos que soma 1. O retorno esperado é `w @ mu`, a variância é `w @ cov @ w` e o Índice de Sharpe divide o prêmio sobre o CDI pela volatilidade. A carteira de pesos iguais serve de primeiro teste: o resultado da forma vetorial tem de bater com o de calcular a série diária da carteira e anualizar.

In [ ]:
def metricas(w, mu=mu.values, cov=cov.values, rf=rf):
    """(retorno, volatilidade, Sharpe) anuais de uma carteira de pesos w."""
    w = np.asarray(w, dtype=float)
    retorno = float(w @ mu)
    volatilidade = float(np.sqrt(w @ cov @ w))
    return retorno, volatilidade, (retorno - rf) / volatilidade


iguais = np.full(len(TICKERS), 1 / len(TICKERS))
ret, vol, sharpe = metricas(iguais)
diaria = retornos.values @ iguais
assert np.isclose(ret, diaria.mean() * PREGOES_ANO)
assert np.isclose(vol, diaria.std(ddof=1) * np.sqrt(PREGOES_ANO))
print("pesos iguais: retorno %.2f%%, volatilidade %.2f%%, Sharpe %.2f" % (100 * ret, 100 * vol, sharpe))
print("ativos com retorno esperado abaixo do CDI:", int((mu < rf).sum()), "de", len(mu))

## 3. O motor vetorizado de Monte Carlo

O motor sorteia `n` linhas de números Uniform(0,1), divide cada linha pela própria soma e calcula as três métricas de todas as linhas de uma vez. `np.einsum("ij,jk,ik->i", pesos, cov, pesos)` é a variância `w @ cov @ w` de cada linha, sem laço em Python. A semente fixa reproduz o mesmo sorteio em qualquer máquina, o que permite escrever um teste com resultado esperado no `SPEC.md`. O requisito do A3 é rodar 10.000 carteiras em menos de 10 segundos.

In [ ]:
def sortear(n, mu=mu.values, cov=cov.values, rf=rf, semente=42):
    """n carteiras com pesos Uniform(0,1) divididos pela soma: (pesos, retorno, volatilidade, sharpe)."""
    gerador = np.random.default_rng(semente)
    pesos = gerador.random((n, len(mu)))
    pesos /= pesos.sum(axis=1, keepdims=True)
    retorno = pesos @ mu
    volatilidade = np.sqrt(np.einsum("ij,jk,ik->i", pesos, cov, pesos))
    return pesos, retorno, volatilidade, (retorno - rf) / volatilidade


inicio = time.perf_counter()
pesos, ret, vol, sharpe = sortear(10_000)
print("10.000 carteiras em %.3f s" % (time.perf_counter() - inicio))
assert np.allclose(pesos.sum(axis=1), 1) and (pesos >= 0).all()
melhor = int(np.argmax(sharpe))
print("abaixo do CDI:", int((sharpe < 0).sum()), "de", len(sharpe))
print("melhor sorteada: Sharpe %.2f, retorno %.2f%%, volatilidade %.2f%%, maior peso %.2f%%"
      % (sharpe[melhor], 100 * ret[melhor], 100 * vol[melhor], 100 * pesos[melhor].max()))
print("maior peso entre todas as sorteadas: %.2f%%" % (100 * pesos.max()))

fig, eixo = plt.subplots(figsize=(10, 5))
pontos = eixo.scatter(vol, ret, c=sharpe, s=4, cmap="viridis")
eixo.scatter(vol[melhor], ret[melhor], s=120, marker="o", edgecolor="black", label="melhor sorteada")
eixo.axhline(rf, linestyle="--", color="gray", label="CDI")
fig.colorbar(pontos, label="Sharpe")
eixo.set_xlabel("volatilidade anual")
eixo.set_ylabel("retorno anual")
eixo.legend()
plt.show()

### Pergunta para a dupla

A melhor sorteada é a melhor carteira possível com estes 15 ativos? A célula abaixo mede a melhor sorteada para números crescentes de sorteios, sempre com a mesma semente. Antes de rodar, anotem quantos sorteios vocês acham que bastam para passar de Sharpe 0,5.

In [ ]:
for n in (10, 100, 1_000, 10_000, 100_000):
    print("%7d sorteios: melhor Sharpe %.2f" % (n, sortear(n)[3].max()))

<details>
<summary>Resposta</summary>

Nenhum desses tamanhos passa de 0,5. A melhor das 10.000 fica em 0,41 e a das 100.000 continua em 0,41; seriam precisos um milhão de sorteios para chegar a 0,60, e a carteira de máximo Sharpe da seção 4 tem 0,92. Parte da distância vem da distribuição dos pesos: somar 15 números Uniform(0,1) e dividir pela soma acumula as carteiras perto de 1/15 por ativo, e nenhuma das 10.000 sorteadas põe mais de 24,61% num único ativo, enquanto a ótima põe 44,31% na PETR4 e zera 12 ativos. A célula seguinte troca a distribuição e mantém os 10.000 sorteios.

</details>

### Distribuição dos pesos com o mesmo orçamento de sorteios

`gerador.random` dividido pela soma da linha não sorteia de modo uniforme no simplex (o conjunto de pesos não negativos que somam 1). A uniforme no simplex é a Dirichlet com todos os parâmetros iguais a 1, e uma Dirichlet com parâmetro abaixo de 1 favorece carteiras concentradas, perto dos vértices. A célula compara as três com a mesma semente e 10.000 sorteios, na cesta de 15 ativos e no trio PETR4, IVVB11 e GOLD11. O número efetivo de ativos é `1 / soma(w**2)`: 15 com pesos iguais e 1 com todo o peso num ativo.

In [ ]:
def sortear_pesos(n, k, alfa=None, semente=42):
    """alfa=None: Uniform(0,1) dividido pela soma, o motor acima; com alfa: Dirichlet simétrica."""
    gerador = np.random.default_rng(semente)
    if alfa is None:
        pesos = gerador.random((n, k))
        return pesos / pesos.sum(axis=1, keepdims=True)
    return gerador.dirichlet(np.full(k, alfa), n)


TRIO = ["PETR4", "IVVB11", "GOLD11"]
for cesta, m, c in (("15 ativos", mu, cov), ("trio", mu[TRIO], cov.loc[TRIO, TRIO])):
    for nome, alfa in (("Uniform(0,1) / soma", None), ("Dirichlet(1)", 1.0), ("Dirichlet(0,2)", 0.2)):
        w = sortear_pesos(10_000, len(m), alfa)
        s = (w @ m.values - rf) / np.sqrt(np.einsum("ij,jk,ik->i", w, c.values, w))
        print("%-9s %-19s melhor Sharpe %.2f | maior peso da melhor %6.2f%% | ativos efetivos (mediana) %5.2f"
              % (cesta, nome, s.max(), 100 * w[s.argmax()].max(), np.median(1 / (w ** 2).sum(axis=1))))

<details>
<summary>Resposta</summary>

Na cesta de 15 ativos, a Dirichlet(1) chega a 0,64 com 10.000 sorteios, acima do milhão de sorteios do motor da aula (0,60), e a Dirichlet(0,2) chega a 0,85. No trio, as três distribuições chegam a 0,92. O resultado depende de três coisas: a distribuição dos pesos, a dimensão do problema e a ausência de objetivo no sorteio, que não usa o Sharpe para decidir onde sortear. Com 15 ativos, nenhuma distribuição chega à ótima, e por isso a seção 4 resolve a carteira com otimização.

</details>

## 4. Otimização com restrições: variância mínima e máximo Sharpe

`scipy.optimize.minimize` com o método SLSQP recebe a função objetivo, a restrição de igualdade (os pesos somam 1) e os limites de cada peso (entre 0 e 1, sem venda a descoberto). Para o máximo Sharpe, minimiza-se o Sharpe com sinal trocado. O resultado só vale com `success` verdadeiro.

In [ ]:
def otimizar(objetivo, n, restricoes=(), limites=None):
    restricoes = [{"type": "eq", "fun": lambda w: w.sum() - 1}] + list(restricoes)
    limites = [(0, 1)] * n if limites is None else limites
    resultado = minimize(objetivo, np.full(n, 1 / n), method="SLSQP", bounds=limites,
                         constraints=restricoes, options={"ftol": 1e-12, "maxiter": 500})
    assert resultado.success, resultado.message
    return resultado.x


M, C = mu.values, cov.values
inicio = time.perf_counter()
w_mv = otimizar(lambda w: w @ C @ w, len(M))
w_ms = otimizar(lambda w: -(w @ M - rf) / np.sqrt(w @ C @ w), len(M))
print("as duas ótimas em %.3f s" % (time.perf_counter() - inicio))

comparacao = pd.DataFrame(
    [metricas(w_mv), metricas(w_ms)],
    index=["variância mínima", "máximo Sharpe"], columns=["retorno", "volatilidade", "sharpe"])
comparacao["ativos com peso"] = [(w_mv > 1e-4).sum(), (w_ms > 1e-4).sum()]
comparacao

In [ ]:
pesos_otimos = pd.DataFrame({"variância mínima": w_mv, "máximo Sharpe": w_ms}, index=TICKERS)
(100 * pesos_otimos[(pesos_otimos > 1e-4).any(axis=1)]).round(2).sort_values("variância mínima", ascending=False)

## 5. A fronteira de 50 pontos sobre a nuvem

Cada ponto da fronteira é a carteira de menor volatilidade para um retorno alvo, com uma restrição a mais: `w @ mu == alvo`. Os 50 alvos vão do retorno da variância mínima até o maior retorno esperado da cesta, onde a carteira vira um ativo só.

In [ ]:
alvos = np.linspace(metricas(w_mv)[0], M.max(), 50)
fronteira = []
for alvo in alvos:
    w = otimizar(lambda w: w @ C @ w, len(M), [{"type": "eq", "fun": lambda w, a=alvo: w @ M - a}])
    fronteira.append(metricas(w)[:2])
fronteira = np.array(fronteira)

fig, eixo = plt.subplots(figsize=(10, 5))
eixo.scatter(vol, ret, s=3, color="lightgray", label="10.000 sorteadas")
eixo.plot(fronteira[:, 1], fronteira[:, 0], linewidth=2.5, label="fronteira de 50 pontos")
for w, rotulo in ((w_mv, "variância mínima"), (w_ms, "máximo Sharpe")):
    r, v, _ = metricas(w)
    eixo.scatter(v, r, s=120, edgecolor="black", zorder=3, label=rotulo)
eixo.scatter(vol[melhor], ret[melhor], s=80, marker="x", color="black", label="melhor sorteada")
eixo.set_xlabel("volatilidade anual")
eixo.set_ylabel("retorno anual")
eixo.legend()
plt.show()

## 6. Desafio para o A3 e o A4

a. Escrever `maximo_sharpe_com_teto`, que resolve o máximo Sharpe com cada peso entre 0 e `PESO_MAXIMO`, trocando só os limites passados ao SLSQP.
b. Comparar os pesos resultantes com os da seção 4: quantos ativos entram, qual é o novo Sharpe e quanto ele cai.
c. Registrar no ADR das restrições da otimização o teto escolhido e o impacto numérico no Sharpe.

In [ ]:
PESO_MAXIMO = 0.30  # teto por ativo do desafio


def maximo_sharpe_com_teto(teto):
    """Desafio: pesos de máximo Sharpe com cada peso entre 0 e teto."""
    raise NotImplementedError("desafio do notebook: troque os limites passados a otimizar")


try:
    w_teto = maximo_sharpe_com_teto(PESO_MAXIMO)
    print(pd.Series(w_teto, index=TICKERS).round(4))
except NotImplementedError as aviso:
    print(aviso)